## Notebook Workflow Structure

This notebook systematically tests bloods-related features extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Environment setup (imports, paths) | Python path configured correctly |
| 3 | Cleanup previous test outputs | No stale data remains |
| 4 | Start Elasticsearch container | Container running on port 9200 |
| 5 | Create credentials file | `test_elastic_credentials.py` generated |
| 6-7 | Populate dummy patient data + bloods data | 5 patients with bloods in Elasticsearch cluster |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logging | Database at `outputs/temp_bloods_db.sqlite` |
| 11-12 | Create pat2vec config with bloods mode | Config object created successfully |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients without errors |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Output features dataframe preview | 3 rows of extracted features displayed |
| 18 | Bloods mode data retrieval test | `bloods_data` contains patient features, non-empty |
| 19-20 | Feature merge functionality | `merge_bloods_csv()` creates CSV file with data |
| 21-22 | Database and project cleanup | All temporary files deleted |
| 23 | Final verification | All assertions pass |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- Empty patient list after population
- Empty DataFrame returned from feature extraction
- Bloods data retrieval returns empty or None result
- Patient count does not match expected (5 patients)
- Merge functionality produces empty result (length 0) - FATAL ERROR
- Cleanup verification fails (residual files remain)

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
for dir_to_remove in ["bloods_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        raise RuntimeError(
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data."
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container (this may take a few seconds)...")
if not es_container.start():
    raise RuntimeError(
        "Failed to start Elasticsearch container. Check if Docker is running."
    )

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

os.makedirs("bloods_test_project", exist_ok=True)
cred_path = os.path.join("bloods_test_project", creds_filename)
with open(cred_path, "w") as f:
    f.write(creds_content)

print(f"Created '{cred_path}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.abspath("../test_files/elastic_schemas.json")
config_populate = config_class(
    proj_name="bloods_test_project",
    credentials_path=cred_path,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
import pandas as pd

# Generate and ingest bloods (basic_observations) data to Elasticsearch
from pat2vec.util.get_dummy_data_cohort_searcher import generate_basic_observations_data
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch

# Generate bloods data for each patient
bloods_dfs = []
for pid in patient_ids:
    df = generate_basic_observations_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    bloods_dfs.append(df)

# Combine all bloods data
df_bloods = (
    pd.concat(bloods_dfs, ignore_index=True) if len(bloods_dfs) > 1 else bloods_dfs[0]
)
df_bloods = df_bloods.where(pd.notnull(df_bloods), None)

# Ingest into Elasticsearch
ingest_data_to_elasticsearch(df_bloods, "basic_observations", es_client=cs.elastic)
cs.elastic.indices.refresh(index="basic_observations")

print(f"Ingested {len(df_bloods)} bloods observations for {len(patient_ids)} patients")

In [ ]:
PROJ_NAME = "bloods_test_project"
DB_FILENAME = "temp_bloods_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    raise RuntimeError(
        f"Failed to remove old database file '{DB_PATH}': {e}. "
        "Critical error - cannot start with stale data."
    ) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=cred_path,
    current_path_dir="",
    main_options={"bloods": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print("pat2vec configuration created with bloods-only sources and database backend.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    ) from e
except ValueError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    ) from e
except RuntimeError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    ) from e
except Exception as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError(
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    raise RuntimeError(
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
from pat2vec.util.post_processing import extract_datetime_to_column

df_with_datetime = extract_datetime_to_column(all_features)

print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {df_with_datetime.shape}")
print(f"Total features: {len(df_with_datetime.columns)}")

if not df_with_datetime.empty:
    print()
    print("First 3 rows:")
    print(df_with_datetime.head(3))
else:
    raise RuntimeError(
        "DataFrame is empty after datetime extraction. Critical error - no features to extract."
    )

In [ ]:
print("\n=== DEMONSTRATING DATA RETRIEVAL FOR BLOODES MODE ===")

all_pat_list = pat2vec_obj.all_patient_list

from pat2vec.pat2vec_get_methods.get_method_bloods import get_current_pat_bloods

# Initialize empty DataFrame if needed
pat_batch = pd.DataFrame()

bloods_data = get_current_pat_bloods(
    current_pat_client_id_code=all_pat_list[0],
    target_date_range=(2020, 1, 1, 2023, 12, 31),
    pat_batch=pat_batch,
    config_obj=config_obj,
)

In [ ]:
if bloods_data is None or (isinstance(bloods_data, list) and len(bloods_data) == 0):
    raise RuntimeError(
        "FATAL ERROR: get_current_pat_bloods returned empty result. "
        "This indicates a critical failure in bloods feature extraction."
    )

if isinstance(bloods_data, list) and len(bloods_data) > 0:
    patient_count = len(bloods_data)
else:
    patient_count = 1

print(f"Retrieved bloods data for {patient_count} patient(s)")
if isinstance(bloods_data, list):
    print(f"\nbloods columns: {list(bloods_data[0].columns)}")
    print("\nSample bloods features:")
    print(bloods_data[0])
else:
    print(f"\nbloods columns: {list(bloods_data.columns)}")
    print("\nSample bloods features:")
    print(bloods_data)

In [ ]:
import pandas as pd

from pat2vec.util.post_processing_build_methods import merge_bloods_csv

print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")
merged_path = merge_bloods_csv(all_pat_list, config_obj, overwrite=True)
merged_data = pd.read_csv(merged_path)
print(f"Merged bloods data saved to: {merged_path}")
print(f"Shape: {merged_data.shape}")
if merged_data.empty:
    raise RuntimeError(
        "FATAL ERROR: Merged bloods dataframe is empty. "
        "This indicates the pat2vec pipeline did not save data to database "
        "or no bloods observations were found for the patients."
    )
print(f"\nColumns: {list(merged_data.columns)}")
print("\nData preview:")
print(merged_data.head())

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete."
    ) from e

try:
    if os.path.exists(cred_path):
        os.remove(cred_path)
        print(f"Removed Elasticsearch credentials: {cred_path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove Elasticsearch credentials file '{cred_path}': {e}. "
        "Critical error - cleanup incomplete."
    ) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(cred_path), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")